# 🎨 TENDOO AI STUDIO - DEMO POSTER THƯƠNG MẠI
### Nền Tảng Tự Động Hóa Thiết Kế Poster Bán Hàng & Quảng Cáo Đa Tầng Bằng AI

Chào mừng **Sếp và các đồng nghiệp** đến với **Tendoo AI Studio**!
- **Kiến trúc**: `FLUX.2 Klein DiT` (Single-pass Regional Velocity Blending) + `HTML5 Sub-pixel Vector Typography`.
- **Phần cứng**: Tối ưu hóa cho **2x NVIDIA A30 (48GB VRAM)**, giữ ấm mô hình trong bộ nhớ để sinh ảnh tức thì **~3.5s / ảnh HD 1024x1024**.
- **Bố cục thương mại hỗ trợ (4 Layouts)**:
  1. 🏛️ **Vòm Đỉnh (Top Dome)**: Đồ uống, chai lọ thẳng đứng, mỹ phẩm trung tâm.
  2. ⏳ **Đồng Hồ Cát (Center Hourglass)**: Bánh trung thu, giỏ quà, hộp quà, vật thể tròn.
  3. 🚘 **Bệ Đáy Điện Ảnh (Bottom Platform)**: Xe hơi, bất động sản, villa, công nghệ cao.
  4. 👗 **Dải Lụa Phân Cột (Split Column)**: Thời trang Lookbook, người mẫu toàn thân, mỹ phẩm.

> **Hướng dẫn sử dụng**:
> 1. Chạy **Cell 1** để nạp mô hình vào 2 GPU (chỉ tốn ~10 giây cho lần đầu tiên).
> 2. Chạy **Cell 2** để hiển thị bảng điều khiển trực quan: bấm các nút mẫu nhanh (1-Click Presets) hoặc tự nhập nội dung, rồi nhấn nút **🚀 Tạo Poster Thương Mại**!
> 3. Chạy **Cell 3** khi kết thúc demo để giải phóng sạch 100% VRAM GPU.

In [ ]:
# ==============================================================================
# BƯỚC 1: NẠP & GIỮ MÔ HÌNH TRONG VRAM TRÊN 2x NVIDIA A30
# ==============================================================================
import os, sys, time, gc
from pathlib import Path

# Cấu hình đường dẫn dự án
PROJECT_ROOT = Path('.').resolve()
for p in [PROJECT_ROOT, PROJECT_ROOT / 'src', PROJECT_ROOT / 'scripts']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
from PIL import Image
import torch

from tendoo.layouts import (
    PosterContent,
    analyze_color_harmony,
    get_layout,
    list_layouts,
)
from tendoo.typography_engine import PosterRenderer

# Biến toàn cục giữ mô hình ấm trong VRAM
global DIT_MODEL, AE_MODEL, TEXT_ENCODER, AE_DTYPE, DEVICE_DIT, DEVICE_AUX

if 'DIT_MODEL' not in globals() or DIT_MODEL is None:
    print('=' * 75)
    print('🚀 ĐANG KHỞI TẠO FLUX.2 KLEIN 4B TRÊN 2x NVIDIA A30 (CHỈ CHẠY 1 LẦN)...')
    print('=' * 75)
    
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        if num_gpus > 1:
            DEVICE_DIT = 'cuda:0'
            DEVICE_AUX = 'cuda:1'
            print(f'  [Phân bổ Multi-GPU]: DiT trên {DEVICE_DIT} | Qwen3 & VAE trên {DEVICE_AUX}')
        else:
            DEVICE_DIT = DEVICE_AUX = 'cuda:0'
            print(f'  [Single-GPU]: Tất cả mô hình trên {DEVICE_DIT}')

        from flux2 import util
        model_name = 'flux.2-klein-4b'

        pdata_candidates = [
            Path(os.path.expanduser('~/persistent-data/FLUX.2-klein-4B')),
            Path(os.path.expanduser('~/persistent-data/FLUX.2-klein-base-4B')),
        ]
        for pdata in pdata_candidates:
            cand = pdata / 'flux-2-klein-4b.safetensors'
            if cand.exists() and 'KLEIN_4B_MODEL_PATH' not in os.environ:
                os.environ['KLEIN_4B_MODEL_PATH'] = str(cand)
                break

        t0 = time.time()
        print('⏳ Đang nạp DiT 4B Flow Model vào GPU 0...')
        DIT_MODEL = util.load_flow_model(model_name, device=DEVICE_DIT)
        DIT_MODEL.eval()

        print('⏳ Đang nạp AutoEncoder (VAE 16x) vào GPU 1...')
        AE_MODEL = util.load_ae(model_name, device=DEVICE_AUX)
        AE_MODEL.eval()
        AE_DTYPE = next(AE_MODEL.parameters()).dtype

        print('⏳ Đang nạp Qwen3 Text Encoder vào GPU 1...')
        TEXT_ENCODER = util.load_text_encoder(model_name, device=DEVICE_AUX)

        dur_init = time.time() - t0
        print(f'\n✅ TẤT CẢ MÔ HÌNH ĐÃ SẴN SÀNG TRONG VRAM ({dur_init:.1f}s)!')
        for dev_id in range(num_gpus):
            alloc = torch.cuda.memory_allocated(dev_id) / (1024 ** 2)
            res = torch.cuda.memory_reserved(dev_id) / (1024 ** 2)
            print(f'  [GPU {dev_id}] Đang chiếm dụng: {alloc:.1f} MB (Dành riêng: {res:.1f} MB)')
        print('=' * 75)
    else:
        print('🟡 Không có GPU CUDA. Chạy ở chế độ MOCK / LOCAL.')
        DIT_MODEL = AE_MODEL = TEXT_ENCODER = None
        DEVICE_DIT = DEVICE_AUX = 'cpu'
else:
    print('⚡ MÔ HÌNH ĐÃ ẤM VÀ SẴN SÀNG TRONG VRAM (Không cần nạp lại)!')

In [ ]:
# ==============================================================================
# BƯỚC 2: BẢNG ĐIỀU KHIỂN TƯƠNG TÁC (TENDOO AI STUDIO UI)
# ==============================================================================
import base64, io, time
import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output

# Catalog 1-Click Presets
PRESETS = {
    'beverage': {
        'layout': 'top_dome',
        'pre_header': 'BỘ SƯU TẬP MÙA HÈ',
        'headline': 'TRÀ ĐÀO CAM SẢ\nTHANH MÁT TỰ NHIÊN',
        'slogan': 'Thưởng thức trọn vẹn từng giọt tươi mát từ thiên nhiên',
        'offer_main': 'MUA 2 TẶNG 1',
        'offer_sub': 'Áp dụng tại mọi chi nhánh trên toàn quốc',
        'brand': 'Tendoo Beverage',
        'hotline': '1900 8888',
        'prompt_scene': 'Commercial beverage advertisement photo of a tall glass of iced peach tea with fresh sliced oranges, lemongrass stalks, floating mint leaves, crystalline water splash, sunlight studio lighting, 8k crisp details, unbranded, zero text'
    },
    'headphones': {
        'layout': 'top_dome',
        'pre_header': 'ÂM THANH ĐỈNH CAO',
        'headline': 'TAI NGHE KHÔNG DÂY\nHI-RES AUDIO',
        'slogan': 'Công nghệ chống ồn chủ động Hybrid ANC thế hệ mới',
        'offer_main': 'GIẢM NGAY 30%',
        'offer_sub': 'Tặng kèm bao da cao cấp trị giá 500.000đ',
        'brand': 'Tendoo Audio Lab',
        'hotline': '0334 842 155',
        'prompt_scene': 'Commercial product photography of sleek matte black premium wireless over-ear headphones on a dark polished slate pedestal, dramatic studio rim light, moody neon blue atmospheric glow, hyper-detailed, clean backdrop, unbranded, zero text'
    },
    'midautumn': {
        'layout': 'center_hourglass',
        'pre_header': 'TẾT TRÔNG TRĂNG ĐOÀN VIÊN',
        'headline': 'HỘP BÁNH TRUNG THU\nHOÀNG GIA THƯỢNG HẠNG',
        'slogan': 'Món quà trọn vẹn ân tình gửi gắm gia đình',
        'offer_main': 'CHIẾT KHẤU ĐẾN 20%',
        'offer_sub': 'Tặng kèm trà sen Tây Hồ hảo hạng',
        'brand': 'Tendoo Mooncake',
        'hotline': '0988 123 456',
        'prompt_scene': 'Traditional Mid-Autumn festival scene with traditional wooden street stalls, glowing lanterns, golden full moon light beam shining down, wooden floorboards, festive atmosphere, unbranded, zero text'
    },
    'suv': {
        'layout': 'bottom_platform',
        'pre_header': 'PHIÊN BẢN GIỚI HẠN 2026',
        'headline': 'KHÁM PHÁ ĐẲNG CẤP\nSUV THẾ HỆ MỚI',
        'slogan': 'Chinh phục mọi cung đường hiểm trở với công nghệ truyền động thông minh',
        'offer_main': 'ƯU ĐÃI 100 TRIỆU',
        'offer_sub': 'Tặng gói bảo hiểm thân vỏ và 3 năm bảo dưỡng miễn phí',
        'brand': 'Tendoo Motors',
        'hotline': '1900 8888',
        'prompt_scene': 'Commercial automobile advertising photography of a sleek luxury metallic dark grey SUV driving on a winding mountain road at golden hour twilight, motion blur wheels, sharp car body reflection, dramatic sky, unbranded, zero text'
    },
    'lookbook': {
        'layout': 'split_column',
        'pre_header': 'BỘ SƯU TẬP THU ĐÔNG 2026',
        'headline': 'ÁO KHOÁC MĂNG TÔ\nDẠ LÔNG CỪU Ý',
        'slogan': 'Chất liệu dạ lông cừu thượng hạng dệt tay, tôn vinh nét thanh lịch vượt thời gian cho quý cô thành thị',
        'offer_main': 'ƯU ĐÃI ĐẾN 25%',
        'offer_sub': 'Tặng khăn choàng lụa tơ tằm cao cấp cho hóa đơn từ 3 triệu',
        'brand': 'Tendoo Atelier',
        'hotline': '1900 6868',
        'prompt_scene': 'Editorial high fashion photography of an elegant Asian female model wearing a tailored luxury wool coat, studio portrait, soft warm rim lighting, dramatic pose, high-end lookbook magazine, unbranded, zero text'
    }
}

# Form Widgets
w_layout = widgets.Dropdown(
    options=[
        ('🏛️ Vòm Đỉnh (Top Arch Dome)', 'top_dome'),
        ('⏳ Đồng Hồ Cát (Center Hourglass)', 'center_hourglass'),
        ('🚘 Bệ Đáy Điện Ảnh (Bottom Platform)', 'bottom_platform'),
        ('👗 Dải Lụa Phân Cột (Split Column)', 'split_column'),
    ],
    value='top_dome',
    description='Bố cục:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='98%')
)

w_kicker = widgets.Text(value='BỘ SƯU TẬP MÙA HÈ', description='Kicker / Phụ:', style={'description_width': '110px'}, layout=widgets.Layout(width='98%'))
w_headline = widgets.Textarea(value='TRÀ ĐÀO CAM SẢ\nTHANH MÁT TỰ NHIÊN', description='Tiêu đề chính:', style={'description_width': '110px'}, layout=widgets.Layout(width='98%', height='64px'))
w_slogan = widgets.Text(value='Thưởng thức trọn vẹn từng giọt tươi mát từ thiên nhiên', description='Slogan:', style={'description_width': '110px'}, layout=widgets.Layout(width='98%'))
w_offer_main = widgets.Text(value='MUA 2 TẶNG 1', description='Ưu đãi Badge:', style={'description_width': '110px'}, layout=widgets.Layout(width='48%'))
w_offer_sub = widgets.Text(value='Áp dụng tại mọi chi nhánh', description='Điều kiện:', style={'description_width': '110px'}, layout=widgets.Layout(width='48%'))
w_brand = widgets.Text(value='Tendoo Beverage', description='Thương hiệu:', style={'description_width': '110px'}, layout=widgets.Layout(width='48%'))
w_hotline = widgets.Text(value='1900 8888', description='Hotline:', style={'description_width': '110px'}, layout=widgets.Layout(width='48%'))
w_prompt = widgets.Textarea(
    value=PRESETS['beverage']['prompt_scene'],
    description='Scene Prompt:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='98%', height='72px')
)

# Nút nạp preset nhanh
btn_bev = widgets.Button(description='🍵 Trà Đào Cam Sả', button_style='info', layout=widgets.Layout(width='auto'))
btn_hp = widgets.Button(description='🎧 Tai Nghe ANC', button_style='info', layout=widgets.Layout(width='auto'))
btn_mid = widgets.Button(description='🥮 Bánh Trung Thu', button_style='info', layout=widgets.Layout(width='auto'))
btn_suv = widgets.Button(description='🏎️ SUV Điện Ảnh', button_style='info', layout=widgets.Layout(width='auto'))
btn_look = widgets.Button(description='👗 Lookbook Thời Trang', button_style='info', layout=widgets.Layout(width='auto'))

def apply_preset(p):
    w_layout.value = p['layout']
    w_kicker.value = p['pre_header']
    w_headline.value = p['headline']
    w_slogan.value = p['slogan']
    w_offer_main.value = p['offer_main']
    w_offer_sub.value = p['offer_sub']
    w_brand.value = p['brand']
    w_hotline.value = p['hotline']
    w_prompt.value = p['prompt_scene']

btn_bev.on_click(lambda _: apply_preset(PRESETS['beverage']))
btn_hp.on_click(lambda _: apply_preset(PRESETS['headphones']))
btn_mid.on_click(lambda _: apply_preset(PRESETS['midautumn']))
btn_suv.on_click(lambda _: apply_preset(PRESETS['suv']))
btn_look.on_click(lambda _: apply_preset(PRESETS['lookbook']))

presets_row = widgets.HBox([widgets.Label('Mẫu 1-Click:'), btn_bev, btn_hp, btn_mid, btn_suv, btn_look], layout=widgets.Layout(margin='0 0 14px 0'))

btn_generate = widgets.Button(
    description='🚀 TẠO POSTER THƯƠNG MẠI (~3.5s)',
    button_style='success',
    layout=widgets.Layout(width='98%', height='48px', margin='16px 0')
)

out_result = widgets.Output()

def on_generate_clicked(b):
    with out_result:
        clear_output()
        print('⏳ Đang khởi chạy FLUX.2 Klein DiT Regional Velocity Blending...')
        t_start = time.time()
        
        layout_name = w_layout.value
        layout = get_layout(layout_name)
        w, h = 1024, 1024
        
        # 1. Mask quang học
        mask_np = layout.generate_mask(width=w, height=h)
        
        # 2. Sinh ảnh nền DiT
        if DIT_MODEL is not None and torch.cuda.is_available():
            from flux2.sampling import get_schedule, prc_img, prc_txt
            from pipeline_e2e_poster import denoise_regional_velocity_blended
            
            prompt_scene = w_prompt.value.strip()
            prompt_corr = layout.get_corridor_prompt('daylight')
            
            with torch.no_grad():
                ctx_scene = TEXT_ENCODER([prompt_scene]).to(torch.bfloat16)
                ctx_scene, ctx_scene_ids = prc_txt(ctx_scene[0])
                ctx_scene = ctx_scene.unsqueeze(0).to(DEVICE_DIT)
                ctx_scene_ids = ctx_scene_ids.unsqueeze(0).to(DEVICE_DIT)

                ctx_corridor = TEXT_ENCODER([prompt_corr]).to(torch.bfloat16)
                ctx_corridor, ctx_corridor_ids = prc_txt(ctx_corridor[0])
                ctx_corridor = ctx_corridor.unsqueeze(0).to(DEVICE_DIT)
                ctx_corridor_ids = ctx_corridor_ids.unsqueeze(0).to(DEVICE_DIT)

                mask_scaled = Image.fromarray(mask_np).resize((w // 16, h // 16), Image.Resampling.BICUBIC)
                mask_flat = torch.from_numpy(np.array(mask_scaled)).float().reshape(1, -1, 1).to(DEVICE_DIT)

                torch.manual_seed(42)
                z_init = torch.randn(1, 128, h // 16, w // 16, device=DEVICE_DIT, dtype=torch.bfloat16)
                img_tokens, img_ids = prc_img(z_init[0])
                img_tokens = img_tokens.unsqueeze(0).to(DEVICE_DIT)
                img_ids = img_ids.unsqueeze(0).to(DEVICE_DIT)

                timesteps = get_schedule(num_steps=8, image_seq_len=img_tokens.shape[1])
                
                z_clean = denoise_regional_velocity_blended(
                    model=DIT_MODEL,
                    img=img_tokens,
                    img_ids=img_ids,
                    txt_scene=ctx_scene,
                    txt_scene_ids=ctx_scene_ids,
                    txt_corridor=ctx_corridor,
                    txt_corridor_ids=ctx_corridor_ids,
                    spatial_mask=mask_flat,
                    timesteps=timesteps,
                    guidance=1.5,
                    num_canvas_tokens=img_tokens.shape[1],
                )
                
                z_dec = z_clean[0].transpose(0, 1).reshape(1, 128, h // 16, w // 16).to(DEVICE_AUX, dtype=AE_DTYPE)
                x_dec = AE_MODEL.decode(z_dec).float()
                x_arr = ((x_dec[0].clamp(-1, 1) + 1) * 127.5).byte().permute(1, 2, 0).cpu().numpy()
                blended_pil = Image.fromarray(x_arr)
        else:
            blended_pil = Image.new('RGB', (w, h), (20, 26, 38))
        
        # 3. Trích xuất màu & Render Typography HTML5
        safe_zone = layout.get_safe_zone()
        palette = analyze_color_harmony(np.array(blended_pil), safe_zone, color_mode='auto')
        
        buf = io.BytesIO()
        blended_pil.save(buf, format='PNG')
        b64_bg = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"
        
        content = PosterContent(
            headline=w_headline.value,
            pre_header=w_kicker.value,
            slogan=w_slogan.value,
            offer_main=w_offer_main.value,
            offer_sub=w_offer_sub.value,
            brand=w_brand.value,
            hotline=w_hotline.value,
        )
        
        html_str = layout.render_html(content=content, palette=palette, bg_data_uri=b64_bg, width=w, height=h)
        
        out_dir = PROJECT_ROOT / 'output_studio_notebook'
        out_dir.mkdir(parents=True, exist_ok=True)
        ts = int(time.time() * 1000)
        out_poster_path = out_dir / f'poster_{ts}.png'
        out_bg_path = out_dir / f'bg_{ts}.png'
        blended_pil.save(out_bg_path)
        
        PosterRenderer.render(html_content=html_str, output_image_path=out_poster_path, width=w, height=h)
        
        duration = time.time() - t_start
        clear_output()
        print(f'⚡ TẠO POSTER THÀNH CÔNG TRONG {duration:.2f} GIÂY!')
        print(f'📁 Đã lưu file: {out_poster_path}')
        
        # Hiển thị ảnh ngay lập tức
        display(IPImage(filename=str(out_poster_path), width=540))

btn_generate.on_click(on_generate_clicked)

# Dựng bố cục Form
row_offers = widgets.HBox([w_offer_main, w_offer_sub])
row_brand = widgets.HBox([w_brand, w_hotline])

form_box = widgets.VBox([
    widgets.HTML('<h3 style="color: #38BDF8; margin: 0 0 10px 0;">⚙️ Thông Tin Poster Khuyến Mại</h3>'),
    presets_row,
    w_layout,
    w_kicker,
    w_headline,
    w_slogan,
    row_offers,
    row_brand,
    w_prompt,
    btn_generate,
    out_result
], layout=widgets.Layout(padding='16px', border='1px solid #243048', border_radius='12px', background_color='#111827', width='100%'))

display(form_box)

In [ ]:
# ==============================================================================
# BƯỚC 3: GIẢI PHÓNG TOÀN BỘ VRAM TRÊN 2 GPU KHI KẾT THÚC DEMO
# ==============================================================================
def free_all_gpu_memory():
    global DIT_MODEL, AE_MODEL, TEXT_ENCODER
    print('🛑 Đang dọn dẹp và giải phóng bộ nhớ GPU...')
    del DIT_MODEL, AE_MODEL, TEXT_ENCODER
    DIT_MODEL = None
    AE_MODEL = None
    TEXT_ENCODER = None
    gc.collect()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
            alloc = torch.cuda.memory_allocated(i) / (1024 ** 2)
            print(f'  [GPU {i}] Đã giải phóng hoàn toàn (Còn lại: {alloc:.1f} MB)')
    print('✨ Toàn bộ VRAM trên cả 2 GPU NVIDIA A30 đã được giải phóng 100%!')

# Chạy giải phóng
free_all_gpu_memory()